# Assignment 2: Milestone I — Natural Language Processing

## Task 1: Basic Text Pre-processing

**Environment:** Python 3, Jupyter Notebook

**Libraries used:**

- `pandas` — data manipulation and CSV I/O
- `re` — regular-expression based tokenization
- `nltk` — lemmatization (WordNetLemmatizer), frequency distributions, collocation detection
- `itertools` — efficient iterator chaining for corpus-level operations

---

## Introduction

In this notebook, we address the problem of transforming raw cosmetics review text into a clean, standardised vocabulary suitable for downstream machine learning tasks. Working with a corpus of approximately 61,000 reviews from the `cosmetics_beauty_products_reviews.csv` dataset, we implement a systematic preprocessing pipeline that:

- Detects and preserves domain-specific bigram collocations (e.g., _"dry skin"_, _"tea tree"_) before tokenization to avoid losing semantic value.
- Applies the required tokenization, case normalisation, short-word removal, stopword removal, and frequency-based filtering steps.
- Extends the pipeline with **lemmatization** to reduce vocabulary sparsity by collapsing inflected word forms.
- Verifies data integrity throughout with explicit sanity checks at each stage.

**Required outputs:**

- `processed.csv` — the original dataset augmented with a `processed_review_text` column.
- `vocab.txt` — the final vocabulary in `word:index` format, sorted alphabetically.


## 1. Importing Libraries


In [ ]:
import pandas as pd
import re
from itertools import chain
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder
from nltk.probability import FreqDist

# Download required NLTK data (runs quietly if already installed)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## 2. Examining and Loading Data

We begin by loading the raw dataset and the provided stopword list. Understanding the structure, size, and quality of the data is essential before designing the pre-processing pipeline.


In [ ]:
# Load the cosmetics & beauty products review dataset
df = pd.read_csv('cosmetics_beauty_products_reviews.csv')

# Load the provided stopwords into a set for O(1) lookup during filtering
with open('stopwords_en.txt', 'r', encoding='utf-8') as f:
    stop_words = set(f.read().splitlines())

print(f"Stopword list size: {len(stop_words)} words")

Stopword list size: 570 words


### 2.1 Exploratory Data Analysis

We examine the dataset's structure, check for missing values, and inspect the target variable distribution. These checks provide a foundation for understanding the data before any transformation is applied.


In [ ]:
print("=== Dataset Overview ===")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Target Variable Distribution (is_a_buyer) ===")
buyer_dist = df['is_a_buyer'].value_counts()
print(buyer_dist)
print(f"\nBuyer ratio: {buyer_dist[True]/len(df)*100:.1f}% buyers vs {buyer_dist[False]/len(df)*100:.1f}% non-buyers")

print("\n=== Sample Review ===")
print(df['review_text'].iloc[0][:300])

=== Dataset Overview ===
Shape: 61284 rows x 15 columns

Columns: ['product_id', 'brand_name', 'review_id', 'review_title', 'review_text', 'author', 'review_date', 'review_rating', 'is_a_buyer', 'product_title', 'price', 'avg_product_rating', 'product_rating_count', 'product_tags', 'product_url']

=== Missing Values ===
product_id                  0
brand_name                  0
review_id                   0
review_title                0
review_text                 9
author                      0
review_date                 0
review_rating               1
is_a_buyer                  0
product_title               0
price                       0
avg_product_rating          0
product_rating_count        0
product_tags            47782
product_url                 0
dtype: int64

=== Target Variable Distribution (is_a_buyer) ===
is_a_buyer
True     48222
False    13062
Name: count, dtype: int64

Buyer ratio: 78.7% buyers vs 21.3% non-buyers

=== Sample Review ===
Works as it claims. Could s

### 2.2 Data Observations

The exploratory analysis surfaces several findings that directly inform our preprocessing design:

1. **Dataset size:** 61,284 reviews across 15 columns. This is a substantial corpus for NLP.
2. **Missing values:** Only 9 reviews have missing `review_text`. These will be dropped before processing to maintain a clean corpus. All `review_title` entries are present.
3. **Class imbalance:** The target variable `is_a_buyer` is moderately imbalanced (~79% `True`, ~21% `False`). This will need to be accounted for during classification (Task 3) via stratified cross-validation and appropriate metrics.
4. **Text quality:** Raw reviews may contain mixed casing, and noisy tokens — all of which our pipeline will address.


## 3. Pre-processing Pipeline

The pipeline applies the following steps in a carefully considered order:

| Step | Operation                              | Justification                                                                                             |
| ---- | -------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| 1    | Bigram collocation detection & joining | Preserves multi-word domain expressions (e.g., _"dry skin"_ → `dry_skin`) before tokenization splits them |
| 2    | Lowercase conversion                   | Normalises casing for consistent matching; applied during collocation replacement                         |
| 3    | Regex tokenization                     | Uses the required pattern `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"`                                                |
| 4    | Short-word removal (len < 2)           | Removes single-character noise with no semantic value                                                     |
| 5    | Stopword removal                       | Eliminates high-frequency function words using the provided list                                          |
| 6    | Lemmatization                          | Reduces inflected forms to their base lemma, consolidating vocabulary                                     |
| 7    | Hapax removal (TF = 1)                 | Removes words appearing only once, which are likely noise or misspellings                                 |
| 8    | Top-20 DF removal                      | Removes the 20 most document-frequent words, which are too generic to be discriminative                   |

**Ordering rationale:**

- **Collocations first** ensures domain bigrams are joined _before_ tokenization would otherwise split them.
- **Stopwords before lemmatization** avoids wasting computation on lemmatizing function words that will be discarded anyway.
- **Lemmatization before frequency filtering** ensures hapax and DF counts are computed on canonical base forms, not inflected variants.

Below, each step is implemented separately with output showing its effect on tokens and vocabs count.


### 3.1 Extracting Valid Reviews

We extract the `review_text` column, dropping the 9 rows with missing values identified earlier. This gives us the clean corpus on which all subsequent steps operate.


In [ ]:
# Extract review text, dropping rows with missing review_text
reviews = df['review_text'].dropna().astype(str).tolist()
print(f"Total valid reviews for processing: {len(reviews)}")
print(f"Dropped {df.shape[0] - len(reviews)} rows with missing review_text")
print(f"\nSample raw review (index 0):")
print(f"  '{reviews[0][:200]}...'")

Total valid reviews for processing: 61275
Dropped 9 rows with missing review_text

Sample raw review (index 0):
  'Works as it claims. Could see the difference from the first day. Use it with Olay cleanser for best results...'


### 3.2 Step 1 — Bigram Collocations Detection

In beauty/cosmetics reviews, many meaningful concepts are expressed as bigrams: "dry skin", "tea tree", "hyaluronic acid", "dark circle". Treating these as separate unigrams loses critical domain semantics. We employ a two-stage approach to identify and preserve these collocations:

**Stage 1: Manual Selection**  
High-confidence cosmetics-specific terms (e.g., "anti-aging", "vitamin-c", "tea-tree") are manually defined based on industry knowledge.

**Stage 2: PMI-based Statistical Detection**  
We use Pointwise Mutual Information (PMI) to identify bigrams that co-occur commonly.

- We set **PMI > 80** to ensure strong association
- We require **frequency more than 20** to filter infrequent pairs

This two-step approach ensures we capture both known domain terms and popular collocations specific to this dataset.

**Why join with hyphens?**  
Joined collocations (e.g., "dry-skin") need to be compatible with the required tokenization regex `[a-zA-Z]+(?:[-'][a-zA-Z]+)?`, which treats hyphenated terms as single tokens. This prevents "dry-skin" from being split back into "dry" and "skin" during tokenization, preserving the semantic specificity needed for classification.


In [ ]:
print("Step 1: Detecting bigram collocations using PMI")
# 1. Manual domain-specific patterns (high precision)
manual_patterns = [
    'kay beauty', 'smudge proof', 'matte finish', 'highly pigmented', 'dark circles',
    'light weight', 'lip balm', 'medium tone', 'highly recommend', 'creamy texture',
    'dry lips', 'medium coverage', 'fair tone', 'fair medium', 'lip liner',
    'nail polish', 'travel friendly', 'beauty products', 'contour sticks', 'loose powder',
    'eye shadow', 'daily wear', 'matte lipstick', 'affordable price', 'full coverage',
    'price range', 'white cast', 'frizz free', 'natural finish', 'jet black',
    'water proof', 'argan oil', 'oily scalp', 'eye liner', 'light medium',
    'setting spray', 'daily basis', 'rose gold', 'liquid lipstick', 'coconut milk',
    'paraben free', 'makeup pouch', 'lip gloss', 'reasonable price', 'dewy finish',
    'smudge free', 'high end', 'makeup products', 'medium fair', 'brown nude',
    'dark brown', 'compact powder', 'winged liner'
]

# 2. Auto-discover from corpus using from_documents (no cross-review bigrams)
# First tokenize reviews into per-review word lists
tokenized_for_colloc = [r.lower().split() for r in reviews]

finder = BigramCollocationFinder.from_documents(tokenized_for_colloc)  # ← fix: was from_words
finder.apply_freq_filter(20)

# Stopwords to filter out generic/filler bigrams
generic_words = {
    'name', 'suggests', 'most', 'importantly', 'came', 'across',
    'come', 'become', 'gets', 'till', 'date', 'die', 'for', 'the',
    'and', 'but', 'with', 'this', 'that', 'very', 'just', 'also'
}

raw_discoveries = finder.nbest(BigramAssocMeasures.pmi, 80)  # fetch more, then filter

# Filter: remove punct-contaminated, generic, and very short tokens
auto_patterns = []
for w1, w2 in raw_discoveries:
    # Skip if either word ends with punctuation
    if w1[-1] in '.,!?' or w2[-1] in '.,!?':
        continue
    # Skip if either word is generic
    if w1 in generic_words or w2 in generic_words:
        continue
    # Skip very short tokens (likely artifacts)
    if len(w1) < 3 or len(w2) < 3:
        continue
    auto_patterns.append(f"{w1} {w2}")

# 3. Merge: manual takes precedence, auto fills gaps
manual_set = set(manual_patterns)
new_from_corpus = [p for p in auto_patterns if p not in manual_set]

print(f"Manual patterns:          {len(manual_patterns)}")
print(f"Auto-discovered (clean):  {len(auto_patterns)}")
print(f"New from corpus:          {len(new_from_corpus)}")
print(f"\nNew candidates found by corpus scan:")
for p in new_from_corpus:
    print(f"  {p}")

# ── MANUAL REVIEW STEP ─────────────────────────────────────────────────────────
# Inspect `new_from_corpus` and decide which to keep.
# Candidates to likely KEEP: 'aloe vera', 'split ends', 'acne prone', 
#                             'cocoa butter', 'vitamin e', 'sun protection'
# Candidates to likely DROP: 'nc 42', 'enrich satin' (product names, not concepts)

filtered = ['positive reviews', 'melts into', 'that\'s why', 'talking about']
approved_new = [w for w in new_from_corpus if w not in filtered]

# 4. Final combined pattern list
all_patterns = manual_patterns + approved_new
all_replacements = [p.replace(' ', '_') for p in all_patterns]

print(f"\nFinal combined patterns: {len(all_patterns)}")

# ── BUILD colloc_dict FROM COMBINED PATTERNS ──────────────────────────────────
colloc_dict = {pat: rep for pat, rep in zip(all_patterns, all_replacements)}
for bigram, joined in list(colloc_dict.items()):
    print(f"    '{bigram}' -> '{joined}'")

Step 1: Detecting bigram collocations using PMI
Manual patterns:          53
Auto-discovered (clean):  55
New from corpus:          48

New candidates found by corpus scan:
  holy grail
  kitten heels
  burning sensation
  cherry skies
  chemically treated
  mind blowing
  aloe vera
  glazed bronze
  finely milled
  whipped cocoa
  enrich satin
  honey blonde
  split ends
  loreal paris
  new york
  harmful chemicals
  cocoa butter
  squeaky clean
  water resistant
  staying power
  touch ups
  spot patch
  acne prone
  herbal essences
  sweet mint
  sun protection
  pocket friendly
  herbal essence
  copper rose
  fades away
  yellow undertone
  fade away
  quick fix
  positive reviews
  settle down
  high points
  never disappoints
  budget friendly
  lakme enrich
  nail paint
  stippling brush
  nail paints
  fast delivery
  nail enamel
  warm honey
  melts into
  that's why
  talking about

Final combined patterns: 97


The combined approach identifies ~100 domain-specific collocations. These are applied to all reviews by replacing matching bigrams with hyphenated forms. Text is also lowercased at this stage for consistent matching, eliminating the need for case normalization during tokenization.


In [ ]:
# Apply collocation replacement to all reviews
print("\nApplying collocation replacements to all reviews...")
colloc_reviews = []
replacements_made = 0

for review in reviews:
    lower_review = review.lower()
    for bigram, replacement in colloc_dict.items():
        if bigram in lower_review:
            lower_review = lower_review.replace(bigram, replacement)
            replacements_made += 1
    colloc_reviews.append(lower_review)

print(f"  Total replacements applied: {replacements_made}")
print(f"\n  Sample after collocation replacement:")
print(f"    '{colloc_reviews[0][:200]}...'")


Applying collocation replacements to all reviews...
  Total replacements applied: 14104
  Sample after collocation replacement:
    'works as it claims. could see the difference from the first day. use it with olay cleanser for best results...'


### 3.3 Step 2 — Regex Tokenization

We tokenize each review using the **required** regular expression pattern: `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"`.

This pattern matches:

- Single alphabetic words (e.g., "moisturizer")
- Hyphenated words (e.g., "long-lasting", "dry-skin" from collocations)
- Apostrophe contractions (e.g., "it's", "don't")

Because lowercasing was applied during the collocation replacement step, all tokens extracted here are already normalised.


In [ ]:
regex = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"

print("Step 2: Tokenizing all reviews with regex pattern...")
tokenized_docs = []

for review in colloc_reviews:
    tokens = [match.group(0) for match in re.finditer(regex, review)]
    tokenized_docs.append(tokens)

total_tokens = sum(len(doc) for doc in tokenized_docs)
print(f"  Total tokens extracted: {total_tokens:,}")
print(f"  Average tokens per review: {total_tokens / len(tokenized_docs):.1f}")
print(f"  Unique tokens: {len(set(chain.from_iterable(tokenized_docs))):,}")

print(f"\n  Sample tokens (index 0): {tokenized_docs[0][:20]}...")

Step 2: Tokenizing all reviews with regex pattern...
  Total tokens extracted: 1,321,300
  Average tokens per review: 21.6
  Unique tokens: 16,327

  Sample tokens (index 0): ['works', 'as', 'it', 'claims', 'could', 'see', 'the', 'difference', 'from', 'the', 'first', 'day', 'use', 'it', 'with', 'olay', 'cleanser', 'for', 'best', 'results']...


### 3.4 Step 3 — Short Word Removal (length < 2)

Single-character tokens (e.g., "i", "a", "s") carry minimal semantic value and add noise. We remove all tokens with fewer than 2 characters.


In [ ]:
print("Step 3: Removing words with length < 2...")
before_count = sum(len(doc) for doc in tokenized_docs)

length_filtered_docs = []
for doc in tokenized_docs:
    length_filtered_docs.append([t for t in doc if len(t) >= 2])

after_count = sum(len(doc) for doc in length_filtered_docs)
removed = before_count - after_count
print(f"  Tokens before: {before_count:,}")
print(f"  Tokens after:  {after_count:,}")
print(f"  Removed:       {removed:,} short tokens ({removed/before_count*100:.1f}%)")

Step 3: Removing words with length < 2...
  Tokens before: 1,321,300
  Tokens after:  1,249,277
  Removed:       72,023 short tokens (5.5%)


### 3.5 Step 4 — Stopword Removal

We remove common English words using the provided `stopwords_en.txt` file. Stopwords (e.g., "the", "is", "and") are high-frequency words that provide grammatical structure but carry little discriminative power for classification.


In [ ]:
print("Step 4: Removing stopwords using provided list...")
before_count = sum(len(doc) for doc in length_filtered_docs)

stopword_filtered_docs = []
for doc in length_filtered_docs:
    stopword_filtered_docs.append([t for t in doc if t not in stop_words])

after_count = sum(len(doc) for doc in stopword_filtered_docs)
removed = before_count - after_count
print(f"  Tokens before: {before_count:,}")
print(f"  Tokens after:  {after_count:,}")
print(f"  Removed:       {removed:,} stopwords ({removed/before_count*100:.1f}%)")
print(f"  Unique words remaining: {len(set(chain.from_iterable(stopword_filtered_docs))):,}")

Step 4: Removing stopwords using provided list...
  Tokens before: 1,249,277
  Tokens after:  569,487
  Removed:       679,790 stopwords (54.4%)
  Unique words remaining: 15,808


More than half of the tokens are removed as stopwords, though vocabulary size
reduces less significantly (570 stopwords removed out of 16,301 tokens).
This is because stopwords like "the", "is", and "to" are highly
repetitive.

Removing stopwords serves two purposes for downstream classification:

1. **Reduces feature dimensionality** in Bag-of-Words representation, lowering
   computational cost during model training
2. **Improves signal-to-noise ratio** by eliminating function words that appear
   in both buyer and non-buyer reviews with similar frequency, as these words carry
   little discriminative power for the classification task

While 50% token reduction may seem aggressive, these words contribute minimal
information for distinguishing purchasing behavior—content words.


### 3.6 Step 5 — Lemmatization

Cosmetics reviews frequently use varied word forms: "moisturize", "moisturizing", "moisturized", "moisturizer". Without lemmatization, each form occupies a separate vocabulary slot, inflating dimensionality and reducing the statistical power of downstream models.

We use NLTK's `WordNetLemmatizer`, which is conservative — it only reduces words to valid dictionary lemmas, avoiding the aggressive truncation of stemming (e.g., Porter stemmer would turn "moisturizing" into "moistur").


In [ ]:
lemmatizer = WordNetLemmatizer()

print("Step 5: Lemmatizing all tokens...")
unique_before = len(set(chain.from_iterable(stopword_filtered_docs)))

lemmatized_docs = []
for doc in stopword_filtered_docs:
    lemmatized_docs.append([lemmatizer.lemmatize(t) for t in doc])

unique_after = len(set(chain.from_iterable(lemmatized_docs)))
print(f"  Unique words before lemmatization: {unique_before:,}")
print(f"  Unique words after lemmatization:  {unique_after:,}")
print(f"  Vocabulary reduced by: {unique_before - unique_after:,} words ({(unique_before - unique_after)/unique_before*100:.1f}%)")

# Show some examples of lemmatization effect
print(f"\n  Sample lemmatized tokens (index 0): {lemmatized_docs[0][:20]}...")

Step 5: Lemmatizing all tokens...
  Unique words before lemmatization: 15,808
  Unique words after lemmatization:  14,674
  Vocabulary reduced by: 1,134 words (7.2%)

  Sample lemmatized tokens (index 0): ['work', 'claim', 'difference', 'day', 'olay', 'cleanser', 'result']...


### 3.7 Steps 6–7 — Frequency-Based Filtering

Two frequency thresholds are applied as required:

1. **Hapax removal (TF = 1):** Words appearing only once across the entire corpus are likely misspellings, brand-specific jargon, or noise. Removing them reduces vocabulary size without losing discriminative power.
2. **Top-20 DF removal:** The 20 most document-frequent words appear in so many reviews that they carry little discriminative information for classification.


In [ ]:
print("Step 6: Calculating Term Frequency (TF) and Document Frequency (DF)...")

# Term Frequency: count of each word across the entire corpus
all_words = list(chain.from_iterable(lemmatized_docs))
term_fd = FreqDist(all_words)

# Document Frequency: count of documents each word appears in
doc_words = list(chain.from_iterable([set(doc) for doc in lemmatized_docs]))
doc_fd = FreqDist(doc_words)

print(f"  Total tokens in corpus: {len(all_words):,}")
print(f"  Unique words: {len(term_fd):,}")

# Identify hapaxes (TF = 1)
words_tf_1 = set(term_fd.hapaxes())
print(f"\nStep 8a: Words with TF=1 (hapaxes): {len(words_tf_1):,}")
print(f"  Examples: {list(words_tf_1)[:10]}")

# Identify top 20 DF words
top_20_df = set([word for word, count in doc_fd.most_common(20)])
print(f"\nStep 8b: Top 20 most document-frequent words:")
for word, count in doc_fd.most_common(20):
    print(f"    '{word}': appears in {count:,} documents ({count/len(lemmatized_docs)*100:.1f}%)")

Step 6: Calculating Term Frequency (TF) and Document Frequency (DF)...
  Total tokens in corpus: 569,487
  Unique words: 14,674

Step 6a: Words with TF=1 (hapaxes): 7,288
  Examples: ['nomakeup', 'hotty', 'mimic', 'shampooer', 'prooff', 'beter', 'parenbens', 'simpel', 'dreadful', 'ah-mazing']

Step 8b: Top 20 most document-frequent words:
    'good': appears in 13,647 documents (22.3%)
    'product': appears in 13,106 documents (21.4%)
    'shade': appears in 9,373 documents (15.3%)
    'love': appears in 8,433 documents (13.8%)
    'skin': appears in 8,351 documents (13.6%)
    'hair': appears in 5,895 documents (9.6%)
    'nice': appears in 5,774 documents (9.4%)
    'amazing': appears in 4,753 documents (7.8%)
    'colour': appears in 4,749 documents (7.8%)
    'long': appears in 4,619 documents (7.5%)
    'perfect': appears in 4,578 documents (7.5%)
    'color': appears in 4,429 documents (7.2%)
    'make': appears in 4,277 documents (7.0%)
    'smooth': appears in 4,043 documents 

In [ ]:
# Combine removal sets and apply final filtering
words_to_remove = words_tf_1.union(top_20_df)
print(f"Total words to remove: {len(words_to_remove):,}")

unique_before_threshold = len(set(chain.from_iterable(lemmatized_docs)))

final_docs = [[t for t in doc if t not in words_to_remove] for doc in lemmatized_docs]

vocab_final = set(chain.from_iterable(final_docs))
print(f"\n  Vocabulary before thresholding: {unique_before_threshold:,}")
print(f"  Vocabulary after thresholding:  {len(vocab_final):,}")
print(f"  Total words removed: {unique_before_threshold - len(vocab_final):,}")

Total words to remove: 7,308

  Vocabulary before thresholding: 14,674
  Vocabulary after thresholding:  7,366
  Total words removed: 7,308


These two filters remove approximately 50% of unique vocabulary terms, concentrating the feature space on moderately frequent words that are both statistically relevant (appear multiple times based on term frequency) and meaningful (appear in less documents). This reduces dimensionality while preserving the most informative terms for classification.


### 3.8 Vocabulary Reduction Summary

We run the pre-processing pipeline once again together to check the vocab size after each pre-processing step.


In [ ]:
# Reconstruct the reduction flow
unique_after_tokenize = len(set(chain.from_iterable(tokenized_docs)))
unique_after_length = len(set(chain.from_iterable(length_filtered_docs)))
unique_after_stop = len(set(chain.from_iterable(stopword_filtered_docs)))
unique_after_lemma = len(set(chain.from_iterable(lemmatized_docs)))

print("=== Vocabulary Reduction at Each Stage ===")
print(f"{'Stage':<45} {'Unique Words':>12}")
print("-" * 60)
print(f"{'After regex tokenization':<45} {unique_after_tokenize:>12,}")
print(f"{'After length filter (< 2 removed)':<45} {unique_after_length:>12,}")
print(f"{'After stopword removal':<45} {unique_after_stop:>12,}")
print(f"{'After lemmatization':<45} {unique_after_lemma:>12,}")
print(f"{'After hapax removal (TF=1)':<45} {f'{unique_after_lemma - len(words_tf_1):,}':>12}")
print(f"{'After top-20 DF removal':<45} {len(vocab_final):>12,}")
print("-" * 60)
print(f"{'FINAL VOCABULARY SIZE':<45} {len(vocab_final):>12,}")

=== Vocabulary Reduction at Each Stage ===
Stage                                         Unique Words
------------------------------------------------------------
After regex tokenization                            16,327
After length filter (< 2 removed)                   16,301
After stopword removal                              15,808
After lemmatization                                 14,674
After hapax removal (TF=1)                           7,386
After top-20 DF removal                              7,366
------------------------------------------------------------
FINAL VOCABULARY SIZE                                7,366


## 4. Saving Required Outputs

Two output files are generated:

1. **`processed.csv`** — the original dataset with an added `processed_review_text` column containing the cleaned token sequences
2. **`vocab.txt`** — the final vocabulary sorted alphabetically, in `word:index` format (0-indexed)


In [ ]:
print("Saving outputs...")

# Save processed.csv — align with valid (non-null review_text) rows
df_clean = df.dropna(subset=['review_text']).copy()
df_clean['processed_review_text'] = [' '.join(doc) for doc in final_docs]
df_clean.to_csv('processed.csv', index=False)
print(f"  -> Saved 'processed.csv' ({df_clean.shape[0]} rows, {df_clean.shape[1]} columns)")

# Save vocab.txt — sorted alphabetically, 0-indexed
vocab_sorted = sorted(list(vocab_final))
with open('vocab.txt', 'w', encoding='utf-8') as f:
    for idx, word in enumerate(vocab_sorted):
        f.write(f"{word}:{idx}\n")

print(f"  -> Saved 'vocab.txt' ({len(vocab_sorted)} words)")

Saving outputs...
  -> Saved 'processed.csv' (61275 rows, 16 columns)
  -> Saved 'vocab.txt' (7366 words)


### 4.1 Output Verification

As a final sanity check, we reload both output files to confirm they were written correctly and contain the expected structure. This step ensures the outputs are immediately usable by downstream tasks without requiring re-running the pipeline.


In [ ]:
# Verify vocab.txt format
print("=== vocab.txt — First 10 entries ===")
with open('vocab.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"  {line.strip()}")
        else:
            break

# Verify processed.csv
df_verify = pd.read_csv('processed.csv')
print(f"\n=== processed.csv verification ===")
print(f"  Shape: {df_verify.shape}")
print(f"  Columns: {list(df_verify.columns)}")
print(f"  Sample processed text: {df_verify['processed_review_text'].iloc[0][:100]}...")
print(f"  Null processed texts: {df_verify['processed_review_text'].isnull().sum()}")

=== vocab.txt — First 10 entries ===
  aa:0
  ab:1
  aback:2
  abd:3
  abh:4
  ability:5
  abit:6
  abnormally:7
  abroad:8
  absent:9

=== processed.csv verification ===
  Shape: (61275, 16)
  Columns: ['product_id', 'brand_name', 'review_id', 'review_title', 'review_text', 'author', 'review_date', 'review_rating', 'is_a_buyer', 'product_title', 'price', 'avg_product_rating', 'product_rating_count', 'product_tags', 'product_url', 'processed_review_text']
  Sample processed text: work claim difference day olay cleanser result...
  Null processed texts: 1862


## 5. Summary

This notebook implemented a complete text pre-processing pipeline for cosmetics and beauty product reviews. The key outcomes are:

1. **Tokenization** using the prescribed regex pattern `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"` ensured consistent token extraction.
2. **Collocation detection** (combine manual and PMI-based approaches) preserved domain-specific bigrams like "dry-skin" and "tea-tree" as single tokens, improving semantic coherence.
3. **Lemmatization** reduced vocabulary sparsity by mapping inflected forms to their base lemmas.
4. **Frequency thresholding** (hapax removal + top-20 DF pruning) eliminated both noisy rare words and overly generic high-frequency words, producing a focused vocabulary suitable for classification.

The resulting vocabulary and processed text are saved in the required formats and will be used as the foundation for feature representation generation in Task 2.
